<a href="https://colab.research.google.com/github/georgegiannakidis/uhart-ms-prep/blob/main/Week1_Day2_Pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Week 1 — Day 2: Pandas Essentials
### UHart MS CS Pre-Arrival Prep · George · 7:00–8:30 PM
---
**Tonight's structure:**
- `7:00–7:05` → Recap NumPy (run the self-check below)
- `7:05–7:55` → Learn Pandas: DataFrames, filtering, groupby, cleaning
- `7:55–8:20` → 10 practice exercises on a real dataset
- `8:20–8:30` → 3 key takeaways + commit to GitHub

> 💡 **Watch first:** [Complete Python Pandas Tutorial 2024 — Keith Galli](https://www.youtube.com/watch?v=2uvysYbKdjM) (watch up to "Filtering" chapter ~1.5 hrs), then come back here.

---
## ⏮️ NumPy Recap (7:00–7:05)

In [1]:
import numpy as np
# Quick warmup — should feel natural from yesterday
arr = np.array([[1,2,3],[4,5,6],[7,8,9]])
print('Shape:', arr.shape)
print('Row sums:', arr.sum(axis=1))
print('Normalized:', np.round((arr - arr.min()) / (arr.max() - arr.min()), 2))
print('✅ NumPy warmup done — moving to Pandas!')

Shape: (3, 3)
Row sums: [ 6 15 24]
Normalized: [[0.   0.12 0.25]
 [0.38 0.5  0.62]
 [0.75 0.88 1.  ]]
✅ NumPy warmup done — moving to Pandas!


---
## 📦 Setup

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print(f'Pandas version: {pd.__version__}')
print('✅ Ready!')

Pandas version: 2.2.2
✅ Ready!


---
## 1️⃣ Creating DataFrames

A **DataFrame** is Pandas' core structure — think of it as a spreadsheet in Python.
- Rows = observations (e.g. one student, one packet, one vulnerability)
- Columns = features (e.g. name, score, severity)

In [3]:
# Create from a dictionary
data = {
    'System':   ['WebApp', 'Database', 'API', 'Auth', 'VPN'],
    'CVSS':     [9.8, 7.2, 4.5, 9.1, 6.8],
    'Severity': ['Critical', 'High', 'Medium', 'Critical', 'Medium'],
    'Patched':  [False, True, True, False, True]
}
df = pd.DataFrame(data)
print(df)
print()
print('Shape:', df.shape)     # (rows, cols)
print('Columns:', df.columns.tolist())
print('Dtypes:\n', df.dtypes)

     System  CVSS  Severity  Patched
0    WebApp   9.8  Critical    False
1  Database   7.2      High     True
2       API   4.5    Medium     True
3      Auth   9.1  Critical    False
4       VPN   6.8    Medium     True

Shape: (5, 4)
Columns: ['System', 'CVSS', 'Severity', 'Patched']
Dtypes:
 System       object
CVSS        float64
Severity     object
Patched        bool
dtype: object


In [4]:
# Exploring a DataFrame — your daily workflow as a GRC analyst
print(df.head(3))       # first 3 rows
print()
print(df.tail(2))       # last 2 rows
print()
print(df.describe())    # stats for numeric columns
print()
print(df.info())        # column types & null counts

     System  CVSS  Severity  Patched
0    WebApp   9.8  Critical    False
1  Database   7.2      High     True
2       API   4.5    Medium     True

  System  CVSS  Severity  Patched
3   Auth   9.1  Critical    False
4    VPN   6.8    Medium     True

           CVSS
count  5.000000
mean   7.480000
std    2.087343
min    4.500000
25%    6.800000
50%    7.200000
75%    9.100000
max    9.800000

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   System    5 non-null      object 
 1   CVSS      5 non-null      float64
 2   Severity  5 non-null      object 
 3   Patched   5 non-null      bool   
dtypes: bool(1), float64(1), object(2)
memory usage: 257.0+ bytes
None


---
## 2️⃣ Accessing Data

Pandas has multiple ways to select data — know these cold.

In [5]:
# Select a single column → returns a Series
cvss_col = df['CVSS']
print('CVSS column:\n', cvss_col)
print('Type:', type(cvss_col))

CVSS column:
 0    9.8
1    7.2
2    4.5
3    9.1
4    6.8
Name: CVSS, dtype: float64
Type: <class 'pandas.core.series.Series'>


In [6]:
# Select multiple columns → returns a DataFrame
subset = df[['System', 'CVSS', 'Severity']]
print(subset)

     System  CVSS  Severity
0    WebApp   9.8  Critical
1  Database   7.2      High
2       API   4.5    Medium
3      Auth   9.1  Critical
4       VPN   6.8    Medium


In [7]:
# .iloc → select by INTEGER position (like NumPy)
print('Row 0:', df.iloc[0])          # first row
print()
print('Rows 1-2, cols 0-1:\n', df.iloc[1:3, 0:2])

Row 0: System        WebApp
CVSS             9.8
Severity    Critical
Patched        False
Name: 0, dtype: object

Rows 1-2, cols 0-1:
      System  CVSS
1  Database   7.2
2       API   4.5


In [8]:
# .loc → select by LABEL (column name / index label)
print(df.loc[0, 'System'])           # single value
print()
print(df.loc[:, 'CVSS':'Severity'])  # slice of columns

WebApp

   CVSS  Severity
0   9.8  Critical
1   7.2      High
2   4.5    Medium
3   9.1  Critical
4   6.8    Medium


---
## 3️⃣ Filtering Data

This is the most-used Pandas skill in GRC — filtering systems by risk, status, compliance.

In [9]:
# Filter rows where CVSS >= 9.0 (Critical)
critical = df[df['CVSS'] >= 9.0]
print('Critical systems:\n', critical)

Critical systems:
    System  CVSS  Severity  Patched
0  WebApp   9.8  Critical    False
3    Auth   9.1  Critical    False


In [10]:
# Multiple conditions — use & (and) | (or), always with parentheses!
high_unpatched = df[(df['CVSS'] >= 7.0) & (df['Patched'] == False)]
print('High/Critical AND unpatched:\n', high_unpatched)

High/Critical AND unpatched:
    System  CVSS  Severity  Patched
0  WebApp   9.8  Critical    False
3    Auth   9.1  Critical    False


In [11]:
# Filter by string value
critical_sev = df[df['Severity'] == 'Critical']
print('Critical severity:\n', critical_sev)

# .isin() — filter by multiple values
hi_or_crit = df[df['Severity'].isin(['Critical', 'High'])]
print('\nHigh or Critical:\n', hi_or_crit)

Critical severity:
    System  CVSS  Severity  Patched
0  WebApp   9.8  Critical    False
3    Auth   9.1  Critical    False

High or Critical:
      System  CVSS  Severity  Patched
0    WebApp   9.8  Critical    False
1  Database   7.2      High     True
3      Auth   9.1  Critical    False


---
## 4️⃣ Adding, Modifying & Renaming Columns

In [12]:
# Add a new column
df['Risk Score'] = df['CVSS'] * 10   # scale to 0-100
df['Days to Patch'] = [30, 0, 0, 45, 0]  # 0 = already patched

# Conditional column with np.where (like IF() in Excel)
df['Action'] = np.where(df['Patched'], 'Monitor', 'REMEDIATE NOW')

print(df)

     System  CVSS  Severity  Patched  Risk Score  Days to Patch         Action
0    WebApp   9.8  Critical    False        98.0             30  REMEDIATE NOW
1  Database   7.2      High     True        72.0              0        Monitor
2       API   4.5    Medium     True        45.0              0        Monitor
3      Auth   9.1  Critical    False        91.0             45  REMEDIATE NOW
4       VPN   6.8    Medium     True        68.0              0        Monitor


In [13]:
# Rename columns
df_renamed = df.rename(columns={'CVSS': 'CVSS Score', 'Risk Score': 'Risk (0-100)'})
print(df_renamed.columns.tolist())

['System', 'CVSS Score', 'Severity', 'Patched', 'Risk (0-100)', 'Days to Patch', 'Action']


---
## 5️⃣ Handling Missing Data

Real-world data always has gaps — knowing how to handle NaN is critical.

In [14]:
# Create a DataFrame with missing values
messy = pd.DataFrame({
    'System': ['A', 'B', 'C', 'D', 'E'],
    'CVSS':   [9.8, None, 4.5, None, 6.8],
    'Owner':  ['Alice', 'Bob', None, 'Dana', None]
})

print('Before cleaning:\n', messy)
print()
print('Null counts:\n', messy.isnull().sum())

Before cleaning:
   System  CVSS  Owner
0      A   9.8  Alice
1      B   NaN    Bob
2      C   4.5   None
3      D   NaN   Dana
4      E   6.8   None

Null counts:
 System    0
CVSS      2
Owner     2
dtype: int64


In [15]:
# Options for handling nulls:
print('Drop rows with any null:\n', messy.dropna())
print()
print('Fill CVSS nulls with mean:\n', messy['CVSS'].fillna(messy['CVSS'].mean()))
print()
print('Fill Owner nulls with Unknown:\n', messy['Owner'].fillna('Unknown'))

Drop rows with any null:
   System  CVSS  Owner
0      A   9.8  Alice

Fill CVSS nulls with mean:
 0    9.800000
1    7.033333
2    4.500000
3    7.033333
4    6.800000
Name: CVSS, dtype: float64

Fill Owner nulls with Unknown:
 0      Alice
1        Bob
2    Unknown
3       Dana
4    Unknown
Name: Owner, dtype: object


---
## 6️⃣ GroupBy & Aggregation

GroupBy = the most powerful Pandas feature. Split data → Apply function → Combine results.

In [16]:
# Back to our vulnerability DataFrame
print('Average CVSS by Severity:')
print(df.groupby('Severity')['CVSS'].mean())

print()
print('Count of systems by Severity:')
print(df.groupby('Severity')['System'].count())

print()
print('Multiple aggregations at once:')
print(df.groupby('Severity')['CVSS'].agg(['mean', 'min', 'max', 'count']))

Average CVSS by Severity:
Severity
Critical    9.45
High        7.20
Medium      5.65
Name: CVSS, dtype: float64

Count of systems by Severity:
Severity
Critical    2
High        1
Medium      2
Name: System, dtype: int64

Multiple aggregations at once:
          mean  min  max  count
Severity                       
Critical  9.45  9.1  9.8      2
High      7.20  7.2  7.2      1
Medium    5.65  4.5  6.8      2


---
## 7️⃣ Sorting & Value Counts

In [18]:
# Sort by CVSS score descending (highest risk first)
print('Sorted by risk (descending):\n', df.sort_values('CVSS', ascending=False))

print()
# value_counts — frequency of each category
print('Severity distribution:\n', df['Severity'].value_counts())
print()
print('Patched distribution:\n', df['Patched'].value_counts())

Sorted by risk (descending):
      System  CVSS  Severity  Patched  Risk Score  Days to Patch         Action
0    WebApp   9.8  Critical    False        98.0             30  REMEDIATE NOW
3      Auth   9.1  Critical    False        91.0             45  REMEDIATE NOW
1  Database   7.2      High     True        72.0              0        Monitor
4       VPN   6.8    Medium     True        68.0              0        Monitor
2       API   4.5    Medium     True        45.0              0        Monitor

Severity distribution:
 Severity
Critical    2
Medium      2
High        1
Name: count, dtype: int64

Patched distribution:
 Patched
True     3
False    2
Name: count, dtype: int64


---
## 8️⃣ Loading Real Data (CSV)

In practice you'll always load data from files, not create DataFrames by hand.

In [19]:
# Download the Titanic dataset — a classic ML practice dataset
import io, urllib.request

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
try:
    titanic = pd.read_csv(url)
    print('Titanic dataset loaded! Shape:', titanic.shape)
    print(titanic.head())
    print()
    print('Missing values:\n', titanic.isnull().sum())
except Exception as e:
    print(f'Could not load from URL. Using sample data instead.')
    titanic = pd.DataFrame({'PassengerId':[1,2,3],'Survived':[0,1,1],
                           'Pclass':[3,1,3],'Age':[22,38,26],'Fare':[7.25,71.28,7.92]})
    print(titanic)

Titanic dataset loaded! Shape: (891, 12)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0      

---
## 🏋️ Practice Exercises (7:55–8:20 PM)

All exercises use the Titanic dataset. Try without hints first!

In [20]:
# Reload the vulnerability DataFrame for exercises 1-5
df = pd.DataFrame({
    'System':   ['WebApp', 'Database', 'API', 'Auth', 'VPN', 'Mail', 'DNS', 'Firewall'],
    'CVSS':     [9.8, 7.2, 4.5, 9.1, 6.8, 8.5, 9.3, 3.2],
    'Severity': ['Critical','High','Medium','Critical','Medium','High','Critical','Low'],
    'Patched':  [False, True, True, False, True, False, False, True],
    'Owner':    ['Alice','Bob','Alice','Carol','Bob','Carol','Alice', None]
})
print(df)

     System  CVSS  Severity  Patched  Owner
0    WebApp   9.8  Critical    False  Alice
1  Database   7.2      High     True    Bob
2       API   4.5    Medium     True  Alice
3      Auth   9.1  Critical    False  Carol
4       VPN   6.8    Medium     True    Bob
5      Mail   8.5      High    False  Carol
6       DNS   9.3  Critical    False  Alice
7  Firewall   3.2       Low     True   None


In [27]:
# ── Exercise 1 ──────────────────────────────────────────────────
# Filter all UNPATCHED systems with CVSS > 8.0
# Then sort by CVSS descending

# YOUR CODE HERE
high_unpatched = df[(df['CVSS'] >= 8.0) & (df['Patched'] == False)].sort_values('CVSS', ascending=False)
print('High AND unpatched:\n', high_unpatched)

High AND unpatched:
    System  CVSS  Severity  Patched  Owner       Priority
0  WebApp   9.8  Critical    False  Alice  P1 - Critical
6     DNS   9.3  Critical    False  Alice  P1 - Critical
3    Auth   9.1  Critical    False  Carol  P1 - Critical
5    Mail   8.5      High    False  Carol      P2 - High


In [29]:
# ── Exercise 2 ──────────────────────────────────────────────────
# Add a column called 'Priority' using these rules:
# CVSS >= 9.0 → 'P1 - Critical'
# CVSS >= 7.0 → 'P2 - High'
# else       → 'P3 - Standard'
# Hint: use np.select() or nested np.where()

# YOUR CODE HERE
conditions = [
    df['CVSS'] >= 9.0,
    df['CVSS'] >= 7.0
]
choices    = ['P1 - Critical', 'P2 - High']
df['Priority'] = np.select(conditions, choices, default='P3 - Standard')

print(df[['System', 'CVSS', 'Priority']])

     System  CVSS       Priority
0    WebApp   9.8  P1 - Critical
1  Database   7.2      P2 - High
2       API   4.5  P3 - Standard
3      Auth   9.1  P1 - Critical
4       VPN   6.8  P3 - Standard
5      Mail   8.5      P2 - High
6       DNS   9.3  P1 - Critical
7  Firewall   3.2  P3 - Standard


In [30]:
# ── Exercise 3 ──────────────────────────────────────────────────
# Group by Owner and compute:
# - Number of systems each person owns
# - Average CVSS score per owner
# Hint: groupby + agg({'col1': 'count', 'col2': 'mean'})

# YOUR CODE HERE
df.groupby('Owner').agg(systems=('System','count'), avg_cvss=("CVSS", 'mean')).round(1)

,systems,avg_cvss
Owner,,
Alice,3,7.9
Bob,2,7.0
Carol,2,8.8
Unassigned,1,3.2


In [31]:
# ── Exercise 4 ──────────────────────────────────────────────────
# Fill the missing Owner value with 'Unassigned'
# Then count how many systems each owner has (including Unassigned)

# YOUR CODE HERE
df['Owner'] = df['Owner'].fillna('Unassigned')
print(df['Owner'].value_counts())

Owner
Alice         3
Bob           2
Carol         2
Unassigned    1
Name: count, dtype: int64


In [32]:
# ── Exercise 5 ──────────────────────────────────────────────────
# Create a summary DataFrame with ONE ROW per Severity level containing:
# - Count of systems
# - Number unpatched
# - Average CVSS
# Hint: multiple groupby + rename + merge, OR agg with a dict

# YOUR CODE HERE
summary_df = df.groupby('Severity').agg(
    total_systems=('System', 'count'),
    unpatched_count=('Patched', lambda x: (~x).sum()),
    avg_cvss=('CVSS', 'mean')
).round(2)

display(summary_df)

,total_systems,unpatched_count,avg_cvss
Severity,,,
Critical,3,3,9.40
High,2,1,7.85
Low,1,0,3.20
Medium,2,0,5.65


In [33]:
# ── Exercises 6-10 use Titanic dataset ──────────────────────────
# (run this first to make sure titanic is loaded)
try:
    titanic.shape
except NameError:
    titanic = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
print('Columns:', titanic.columns.tolist())

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [35]:
# ── Exercise 6 ──────────────────────────────────────────────────
# What was the survival rate (%) for each passenger class (Pclass)?
# Hint: groupby('Pclass')['Survived'].mean() * 100

# YOUR CODE HERE
survival_rate = titanic.groupby('Pclass')['Survived'].mean() * 100
print(survival_rate)

Pclass
1    62.962963
2    47.282609
3    24.236253
Name: Survived, dtype: float64


In [36]:
# ── Exercise 7 ──────────────────────────────────────────────────
# How many passengers were in each class AND survived?
# Show as a pivot table with Pclass as rows, Survived as columns
# Hint: pd.crosstab()

# YOUR CODE HERE
pd.crosstab(titanic['Pclass'], titanic['Survived'])

Survived,0,1
Pclass,,
1,80,136
2,97,87
3,372,119


In [38]:
# ── Exercise 8 ──────────────────────────────────────────────────
# Fill missing Age values with the MEDIAN age
# Then show the before/after null count for the Age column

# YOUR CODE HERE
titanic['Age'] = titanic['Age'].fillna(titanic['Age'].median())
print('Null counts:\n', titanic['Age'].isnull().sum())

Null counts:
 0


In [39]:
# ── Exercise 9 ──────────────────────────────────────────────────
# Find the top 5 most expensive fares in the dataset
# Show only: Name, Pclass, Fare, Survived

# YOUR CODE HERE
top_fares = titanic.nlargest(5, 'Fare')
top_fares[['Name', 'Pclass', 'Fare', 'Survived']]

,Name,Pclass,Fare,Survived
258,"Ward, Miss. Anna",1,512.3292,1
679,"Cardeza, Mr. Thomas Drake Martinez",1,512.3292,1
737,"Lesurer, Mr. Gustave J",1,512.3292,1
27,"Fortune, Mr. Charles Alexander",1,263.0000,0
88,"Fortune, Miss. Mabel Helen",1,263.0000,1


In [40]:
# ── Exercise 10 — GRC Bonus ──────────────────────────────────────
# Create a 'Risk Category' column on the vuln DataFrame:
# - Unpatched + CVSS >= 9 → 'IMMEDIATE ACTION'
# - Unpatched + CVSS >= 7 → 'URGENT'
# - Patched OR CVSS < 7   → 'MONITORED'
# Then use value_counts() to show the risk distribution
# And print total number of systems needing immediate or urgent action

df = pd.DataFrame({
    'System':  ['WebApp','Database','API','Auth','VPN','Mail','DNS','Firewall'],
    'CVSS':    [9.8, 7.2, 4.5, 9.1, 6.8, 8.5, 9.3, 3.2],
    'Patched': [False, True, True, False, True, False, False, True]
})

# YOUR CODE HERE
conditions = [
    (~df['Patched']) & (df['CVSS'] >= 9.0),
    (~df['Patched']) & (df['CVSS'] >= 7.0)
]
choices = ['IMMEDIATE ACTION', 'URGENT']
df['Risk Category'] = np.select(conditions, choices, default='MONITORED')

print('Risk Distribution:')
print(df['Risk Category'].value_counts())

action_needed = df[df['Risk Category'].isin(['IMMEDIATE ACTION', 'URGENT'])]
print(f'\nTotal systems needing action: {len(action_needed)}')
display(df)

Risk Distribution:
Risk Category
MONITORED           4
IMMEDIATE ACTION    3
URGENT              1
Name: count, dtype: int64

Total systems needing action: 4


,System,CVSS,Patched,Risk Category
0,WebApp,9.8,False,IMMEDIATE ACTION
1,Database,7.2,True,MONITORED
2,API,4.5,True,MONITORED
3,Auth,9.1,False,IMMEDIATE ACTION
4,VPN,6.8,True,MONITORED
5,Mail,8.5,False,URGENT
6,DNS,9.3,False,IMMEDIATE ACTION
7,Firewall,3.2,True,MONITORED


### Tomorrow:
**Day 3 — Matplotlib & Scikit-learn** · Watch Corey Schafer Matplotlib Parts 1, 2, 6, 7 first

In [22]:
# Answer Key — only after attempting!
import numpy as np, pandas as pd

df = pd.DataFrame({
    'System':  ['WebApp','Database','API','Auth','VPN','Mail','DNS','Firewall'],
    'CVSS':    [9.8, 7.2, 4.5, 9.1, 6.8, 8.5, 9.3, 3.2],
    'Severity':['Critical','High','Medium','Critical','Medium','High','Critical','Low'],
    'Patched': [False, True, True, False, True, False, False, True],
    'Owner':   ['Alice','Bob','Alice','Carol','Bob','Carol','Alice', None]
})

# Ex 1
print('Ex1:\n', df[(~df['Patched']) & (df['CVSS'] > 8.0)].sort_values('CVSS', ascending=False))

# Ex 2
conditions = [df['CVSS'] >= 9.0, df['CVSS'] >= 7.0]
choices    = ['P1 - Critical', 'P2 - High']
df['Priority'] = np.select(conditions, choices, default='P3 - Standard')
print('Ex2 Priority column:\n', df[['System','CVSS','Priority']])

# Ex 3
print('Ex3:\n', df.groupby('Owner').agg(systems=('System','count'), avg_cvss=('CVSS','mean')).round(1))

# Ex 4
df['Owner'] = df['Owner'].fillna('Unassigned')
print('Ex4:\n', df['Owner'].value_counts())

# Ex 10 (GRC)
df2 = pd.DataFrame({'System':['WebApp','Database','API','Auth','VPN','Mail','DNS','Firewall'],
                    'CVSS':[9.8,7.2,4.5,9.1,6.8,8.5,9.3,3.2],
                    'Patched':[False,True,True,False,True,False,False,True]})
c2 = [(~df2['Patched']) & (df2['CVSS'] >= 9), (~df2['Patched']) & (df2['CVSS'] >= 7)]
ch2 = ['IMMEDIATE ACTION', 'URGENT']
df2['Risk Category'] = np.select(c2, ch2, default='MONITORED')
print('Ex10:\n', df2['Risk Category'].value_counts())
urgent_count = (df2['Risk Category'].isin(['IMMEDIATE ACTION','URGENT'])).sum()
print('Systems needing action:', urgent_count)

Ex1:
    System  CVSS  Severity  Patched  Owner
0  WebApp   9.8  Critical    False  Alice
6     DNS   9.3  Critical    False  Alice
3    Auth   9.1  Critical    False  Carol
5    Mail   8.5      High    False  Carol
Ex2 Priority column:
      System  CVSS       Priority
0    WebApp   9.8  P1 - Critical
1  Database   7.2      P2 - High
2       API   4.5  P3 - Standard
3      Auth   9.1  P1 - Critical
4       VPN   6.8  P3 - Standard
5      Mail   8.5      P2 - High
6       DNS   9.3  P1 - Critical
7  Firewall   3.2  P3 - Standard
Ex3:
        systems  avg_cvss
Owner                   
Alice        3       7.9
Bob          2       7.0
Carol        2       8.8
Ex4:
 Owner
Alice         3
Bob           2
Carol         2
Unassigned    1
Name: count, dtype: int64
Ex10:
 Risk Category
MONITORED           4
IMMEDIATE ACTION    3
URGENT              1
Name: count, dtype: int64
Systems needing action: 4
